# 02 Figures: one per headline finding

Reads saved results only (no training). Writes PNGs to `docs/figures/`.

Figures 1 to 4 compare the stage 1 and 2 models on two periods: **validation**, which starts the
day after training ends (`results/stage2/report_*`), and **test**, which starts about 40 days
later (`results/final/final_*`, evaluated once). Figures 5 and 6 come from the stage 3 test replay.
Colors follow the model family in every figure: MLP blue, logistic regression orange, LightGBM
aqua. Solid = non-private, dashed = DP.

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

if Path.cwd().name == "notebooks":
    os.chdir("..")
OUT = Path("docs/figures")
OUT.mkdir(parents=True, exist_ok=True)

# Validated categorical palette, fixed order (slots 1-3), plus ink and surface tokens.
MLP, LR, LGBM = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#ebeae6", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlelocation": "left",
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "font.size": 10, "lines.linewidth": 2, "lines.markersize": 7, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
DOT = dict(marker="o", markeredgecolor=SURFACE, markeredgewidth=2, capsize=3)
PERIODS = [("val", "Validation: right after training"), ("test", "Test: about 40 days later")]


def latest(root, prefix):
    pat = re.compile(rf"^{re.escape(prefix)}_\d{{8}}-\d{{6}}(_final)?$")
    dirs = sorted(d for d in Path(root).iterdir() if d.is_dir() and pat.match(d.name))
    if not dirs:
        raise FileNotFoundError(f"no run {prefix} in {root}")
    return dirs[-1]


def points(run_dir):
    return [json.loads((p / "metrics.json").read_text()) for p in sorted(run_dir.iterdir())
            if p.is_dir() and (p / "metrics.json").exists()]


def load(model, period):
    """Stage 2 report (validation) or final evaluation (test); DP runs sorted by epsilon."""
    d = latest("results/stage2", f"report_{model}") if period == "val" else latest("results/final", f"final_{model}")
    return sorted(points(d), key=lambda m: m["params"].get("target_epsilon", 0))


R2 = {(m, p): load(m, p) for m in ("lgbm", "mlp", "logreg", "dp_mlp", "dp_logreg") for p, _ in PERIODS}


def s(m, key):
    return m["summary"][key]["mean"], m["summary"][key]["std"]


def seg_mean(m, field, b="0.02"):
    return [np.mean([m["per_seed"][k][f"budget_{b}"]["segments"][sg][field] for k in m["per_seed"]])
            for sg, _ in SEGS]


def headline(fig, title, sub, top=0.9):
    """Figure title (the finding) and subtitle (data, seeds), above all panels."""
    fig.suptitle(title, x=0.01, y=0.995, ha="left", va="top", fontweight="bold", fontsize=12)
    fig.text(0.01, 0.925, sub, ha="left", va="top", color=INK2, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, top))


SEGS = [("seen_card", "card seen\nin training"), ("new_card", "new card"),
        ("no_key", "no card key\n(e.g. no address)")]

## 1. The 1% budget can hide differences

In [ ]:
budgets = [0.005, 0.01, 0.02]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
for ax, (period, label) in zip(axes, PERIODS):
    at1 = {}
    ax.axvspan(0.9, 1.1, color=GRID, zorder=0)
    for model, color, name in [("lgbm", LGBM, "LightGBM"), ("mlp", MLP, "MLP")]:
        m = R2[model, period][0]
        y = [s(m, f"budget_{b}.recall_share_of_ceiling")[0] for b in budgets]
        e = [s(m, f"budget_{b}.recall_share_of_ceiling")[1] for b in budgets]
        ax.errorbar([b * 100 for b in budgets], y, yerr=e, color=color, **DOT)
        ax.annotate(name, (2, y[-1]), xytext=(9, 0), textcoords="offset points", va="center",
                    color=INK, fontsize=9)
        at1[name] = y[1]
    ax.text(1, 0.61, f"at 1%:\nLightGBM {at1['LightGBM']:.2f}\nMLP {at1['MLP']:.2f}", ha="center",
            va="bottom", color=INK, fontsize=8.5)
    ax.set_xticks([0.5, 1, 2], ["0.5%", "1%", "2%"])
    ax.set_xlim(0.3, 2.65)
    ax.set_xlabel("alert budget (share of transactions reviewed)")
    ax.set_title(label)
axes[0].set_ylabel("share of the best possible recall")
headline(fig, "Right after training, the MLP tied LightGBM at a 1% budget; weeks later it did not",
         "Mean of 3 seeds (bars: std); shaded: the 1% budget. 1.0 = every alert is a fraud.")
fig.savefig(OUT / "01_budget_hides_differences.png")
plt.show()

## 2. LightGBM's edge: recognizing cards it has seen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
x, w = np.arange(len(SEGS)), 0.36
for ax, (period, label) in zip(axes, PERIODS):
    for i, (model, color, name) in enumerate([("lgbm", LGBM, "LightGBM"), ("mlp", MLP, "MLP")]):
        vals = seg_mean(R2[model, period][0], "recall")
        bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.03, color=color, label=name)
        for bx, v in zip(bars, vals):
            ax.text(bx.get_x() + bx.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", va="bottom",
                    color=INK, fontsize=9)
    ax.set_xticks(x, [lab for _, lab in SEGS])
    ax.set_ylim(0, 0.8)
    ax.set_title(label)
axes[0].set_ylabel("recall at 2% budget")
axes[0].legend(loc="upper left")
headline(fig, "LightGBM is the only model that still recognizes known cards weeks later",
         "2% budget, mean of 3 seeds. On new cards the two models are level; on test LightGBM "
         "also catches more keyless frauds.")
fig.savefig(OUT / "02_lightgbm_lead_is_card_memory.png")
plt.show()

## 3. The cost of privacy vs epsilon (2% budget)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.9), sharey=False)
key = "budget_0.02.recall_share_of_ceiling"
for ax, (period, label) in zip(axes, PERIODS):
    for model, color in [("mlp", MLP), ("logreg", LR), ("lgbm", LGBM)]:
        ax.axhline(s(R2[model, period][0], key)[0], color=color, linestyle=":", linewidth=1.6)
    for model, color in [("dp_mlp", MLP), ("dp_logreg", LR)]:
        runs = R2[model, period]
        eps = [m["params"]["target_epsilon"] for m in runs]
        ax.errorbar(eps, [s(m, key)[0] for m in runs], yerr=[s(m, key)[1] for m in runs],
                    color=color, linestyle="--", **DOT)
    ax.set_xscale("log")
    ax.set_xticks(eps, [str(e) for e in eps])
    ax.minorticks_off()
    ax.set_xlim(0.4, 10)
    ax.set_xlabel("privacy budget epsilon (smaller = more private)")
    ax.set_title(label)
axes[0].set_ylabel("share of the best possible recall (2% budget)")
legend = [(MLP, "--", "DP MLP"), (LR, "--", "DP log. reg."), (MLP, ":", "MLP, no DP"),
          (LR, ":", "Log. reg., no DP"), (LGBM, ":", "LightGBM, no DP")]
fig.legend([Line2D([], [], color=c, linestyle=l, marker="o" if l == "--" else None,
                   markeredgecolor=SURFACE) for c, l, _ in legend],
           [n for *_, n in legend], loc="lower center", ncol=5, bbox_to_anchor=(0.5, 0.0),
           fontsize=9, handlelength=2.8)
headline(fig, "Privacy costs the MLP recall in both periods; under DP the simple model does as well",
         "2% budget, mean of 3 fresh seeds (bars: std). Dotted lines: non-private references. "
         "Y axes differ: the test period is harder for every model.", top=0.91)
fig.subplots_adjust(bottom=0.24)
fig.savefig(OUT / "03_privacy_cost_vs_epsilon.png")
plt.show()

## 4. DP erases card memory, but that memory fades within weeks

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.6))
series = [("mlp", None, MLP, "MLP, no DP"), ("dp_mlp", 8, "#86b6ef", "DP MLP, epsilon 8"),
          ("dp_mlp", 1, "#184f95", "DP MLP, epsilon 1")]
x, w = np.arange(len(PERIODS)), 0.26
for i, (model, e, color, name) in enumerate(series):
    vals, counts = [], []
    for period, _ in PERIODS:
        runs = R2[model, period]
        m = runs[0] if e is None else next(r for r in runs if r["params"]["target_epsilon"] == e)
        caught = seg_mean(m, "frauds_flagged")[0]
        total = m["per_seed"][next(iter(m["per_seed"]))]["budget_0.02"]["segments"]["seen_card"]["n_fraud"]
        vals.append(caught / total)
        counts.append((caught, total))
    bars = ax.bar(x + (i - 1) * w, vals, w - 0.03, color=color, label=name)
    for bx, v, (c, t) in zip(bars, vals, counts):
        ax.text(bx.get_x() + bx.get_width() / 2, v + 0.002, f"{c:.0f} of {t}", ha="center",
                va="bottom", color=INK, fontsize=8.5)
ax.set_xticks(x, [lab for _, lab in PERIODS])
ax.set_ylabel("recall on frauds from cards seen in training\n(2% budget)")
ax.legend(loc="upper right")
headline(fig, "DP erases memory of known cards, but that memory fades within weeks anyway",
         "Frauds caught on cards that appeared in training, mean of 3 seeds. Weeks later even the "
         "non-private model catches almost none.")
fig.savefig(OUT / "04_card_memory_fades.png")
plt.show()

## 5. Privacy takes away the benefit of retraining (stage 3, test replay)

In [ ]:
R = latest("results/stage3", "replay_final")
sched = ["none", "expanding_every3", "expanding_every2", "expanding_every1"]
labels = ["never", "every\n3 weeks", "every\n2 weeks", "weekly"]


def curve(exp):
    out = [json.loads((R / exp / sc / "metrics.json").read_text())["summary"]["select_score"] for sc in sched]
    return [o["mean"] for o in out], [o["std"] for o in out]


fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
panels = [
    (axes[0], "Neural network (MLP)", [("mlp", MLP, "-", "no DP"), ("dp_mlp_eps8", MLP, "--", "DP, total epsilon 8")]),
    (axes[1], "Logistic regression", [("logreg", LR, "-", "no DP"), ("dp_logreg_eps8", LR, "--", "DP, total epsilon 8"),
                                      ("dp_logreg_eps2", "#a8461c", "-.", "DP, total epsilon 2")]),
]
for ax, title, series in panels:
    for exp, color, ls, name in series:
        y, e = curve(exp)
        ax.errorbar(range(4), y, yerr=e, color=color, linestyle=ls, **DOT)
        ax.annotate(f"{y[-1] - y[0]:+.3f}", (3, y[-1]), xytext=(9, 0), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    ax.set_xticks(range(4), labels)
    ax.set_xlim(-0.3, 3.75)
    ax.set_xlabel("retraining frequency (expanding window)")
    ax.set_title(title)
    ax.legend(handles=[Line2D([], [], color=c, linestyle=l, marker="o", markeredgecolor=SURFACE)
                       for _, c, l, _ in series],
              labels=[n for *_, n in series], loc="upper left", handlelength=3)
axes[0].set_ylabel("score (share of ceiling, 1% and 2%)")
headline(fig, "Without privacy, retraining pays off; with DP most of the gain disappears",
         "Test period replayed week by week, 14-day label delay, mean of 3 seeds (bars: std). "
         "Labels: weekly minus never.", top=0.91)
fig.savefig(OUT / "05_privacy_erases_retraining_gain.png")
plt.show()

## 6. Drift monitoring missed the performance swings (stage 3, test replay)

In [ ]:
drift = json.loads((R / "drift.json").read_text())
psi_mean = [w["mean_feature_psi"] for w in drift["by_week"]]
never = json.loads((R / "mlp" / "none" / "metrics.json").read_text())["per_seed"]
weekly_share = [np.mean([never[k]["per_week"][w]["budget_0.02"]["recall_share_of_ceiling"] for k in never])
                for w in range(len(psi_mean))]
weeks = np.arange(1, len(psi_mean) + 1)
fig, axes = plt.subplots(2, 1, figsize=(6.8, 5.8), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
axes[0].plot(weeks, weekly_share, color=MLP, marker="o", markeredgecolor=SURFACE, markeredgewidth=2)
axes[0].set_ylabel("share of best possible\nrecall (2% budget)")
axes[1].plot(weeks, psi_mean, color=INK2, marker="o", markeredgecolor=SURFACE, markeredgewidth=2)
for t, lab in [(0.1, "0.1: moderate shift"), (0.25, "0.25: large shift")]:
    axes[1].axhline(t, color=MUTED, linestyle=":", linewidth=1.2)
    axes[1].annotate(lab, (len(weeks) + 0.25, t + 0.006), ha="right", va="bottom", color=INK2, fontsize=8.5)
axes[1].set_ylim(0, 0.3)
axes[1].set_ylabel("mean feature PSI\nvs training data")
axes[1].set_xlabel("week of the test period")
axes[1].set_xticks(weeks)
axes[1].set_xlim(0.7, len(weeks) + 0.3)
headline(fig, "Performance swung week to week while input drift stayed flat",
         "Test period, non-private MLP never retrained, mean of 3 seeds.", top=0.92)
fig.savefig(OUT / "06_drift_monitoring_missed_it.png")
plt.show()